# Train with MLflow tracking

Goal:

- Train a YOLO model to detect license plates.
- Track parameters, metrics and artifacts in the SageMaker MLflow tracking server.
- Export the trained model to `s3://<bucket>/trains/models/`.


## Environment

Install libraries.


In [1]:
%pip install -q -U ultralytics torch torchvision onnx onnxruntime onnxslim mlflow sagemaker-mlflow

Note: you may need to restart the kernel to use updated packages.


Inspect environment.


In [2]:
import os
import sys
from importlib.metadata import version
from pathlib import Path

import torch
import torchvision
import ultralytics

from sagemaker.core.helper.session_helper import Session, get_execution_role

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

# define path
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
RUNS = ROOT / "runs"
MODELS = ROOT / "models"

# create dir
for d in (RAW, PROCESSED, RUNS, MODELS):
    d.mkdir(parents=True, exist_ok=True)

session = Session()
REGION = session.boto_region_name
ROLE = get_execution_role()

# env_file
env_file = Path.home() / ".sagemaker-yolo.env"
if "BUCKET" not in os.environ and env_file.exists():
    for line in env_file.read_text().splitlines():
        key, _, val = line.partition("=")
        os.environ.setdefault(key.strip(), val.strip())

# get bucket id
BUCKET = os.environ["BUCKET"]

# bucket keys
S3_RAW = f"s3://{BUCKET}/raw-data/"
S3_SPLIT = f"s3://{BUCKET}/notebook/split-data"
S3_MODELS = f"s3://{BUCKET}/notebook/models"

# get device
DEVICE = 0 if torch.cuda.is_available() else "cpu"
# device tag
device_tag = "gpu" if torch.cuda.is_available() else "cpu"

# print environment info
print("python         ", sys.version.split()[0])
print("torch          ", torch.__version__)
print("torchvision    ", torchvision.__version__)
print("ultralytics    ", ultralytics.__version__)
print("sagemaker-core ", version("sagemaker-core"))
print("cuda           ", torch.cuda.is_available())
print("device         ", device_tag)
print("region         ", REGION)
print("bucket         ", BUCKET)
print("root           ", ROOT)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


python          3.12.13
torch           2.13.0+cu130
torchvision     0.28.0+cu130
ultralytics     8.4.119
sagemaker-core  2.16.0
cuda            True
device          gpu
region          ca-central-1
bucket          sagemaker-yolo-dev-up68ac
root            /home/sagemaker-user/sagemaker-yolo


## Configure MLflow server


In [3]:
import boto3
import mlflow

from notebook.code.tracking import tracking_uri

# set tracking server
TRACKING_URI = tracking_uri()
mlflow.set_tracking_uri(TRACKING_URI)

# create experiment
EXPERIMENT = "yolo-plate-detection"
experiment = mlflow.set_experiment(EXPERIMENT)

# get tracking server
server = boto3.client("sagemaker").describe_mlflow_tracking_server(
    TrackingServerName=TRACKING_URI.rsplit("/", 1)[-1]
)
# get tracking server url
UI_URL = server["TrackingServerUrl"]

# print mlflow server
print("tracking  ", TRACKING_URI)
print("status    ", server["TrackingServerStatus"])
print("experiment", EXPERIMENT, f"(id {experiment.experiment_id})")

# list experiments to confirm connection
print("\nexisting experiments")
for exp in mlflow.search_experiments():
    print(f"  {exp.experiment_id:>4}  {exp.name}")

# print link and experiments
print(f"\nUI: {UI_URL}/#/experiments/{experiment.experiment_id}")

tracking   arn:aws:sagemaker:ca-central-1:099139718958:mlflow-tracking-server/sagemaker-yolo-dev
status     Created
experiment yolo-plate-detection (id 1)

existing experiments
     1  yolo-plate-detection
     0  Default

UI: https://t-dwc5gdj0ow9o.ca-central-1.experiments.sagemaker.aws/#/experiments/1


## Data processing

Pull the raw data from S3, split it, push the split back.


In [4]:
from notebook.code.data_loader import build_split, summarize, verify_split, write_data_yaml
from notebook.code.s3_sync import download, upload

# e.g. 100 for a fast smoke run
# LIMIT = 100
LIMIT = None  # all images

SPLIT_SEED = 0  # split random seed

# download data
print(download(S3_RAW, RAW))

# inspect and print summary
stats = summarize(RAW)
print({k: stats[k] for k in ("pairs", "boxes_total", "boxes_per_image_max", "malformed")})

# split
print(build_split(RAW, PROCESSED, val_fraction=0.2, limit=LIMIT, seed=SPLIT_SEED))
print(verify_split(PROCESSED))

# save split in s3
print(upload(PROCESSED, S3_SPLIT, delete=True))

# create data config yaml file
names = (RAW / "classes.txt").read_text().split()
data_yaml = write_data_yaml(ROOT / "configs" / "data.yaml", PROCESSED, names)
print(data_yaml.read_text())

{'downloaded': 0, 'skipped': 1113}


{'pairs': 556, 'boxes_total': 574, 'boxes_per_image_max': 3, 'malformed': []}


{'train': 445, 'val': 111, 'orphan_images': 0, 'orphan_labels': 0}
{'train': 445, 'val': 111}


{'uploaded': 0, 'skipped': 1112, 'removed': 0}
path: /home/sagemaker-user/sagemaker-yolo/data/processed
train: train/images
val: val/images
nc: 1
names: ['car_plate']



## Train with tracking

Track by built-in MLflow callback, logging hyperparameters, per-epoch metrics and the run artifacts on its own.

- `MLFLOW_KEEP_RUN_ACTIVE` holds the run open after training.


In [5]:
import time
from ultralytics import YOLO

from notebook.code.data_loader import build_train_cfg

# create train parameters
train_cfg = build_train_cfg(
    device=DEVICE,
    workers=(os.cpu_count() or 2) if DEVICE != "cpu" else 0,
)
train_cfg["project"] = str(ROOT / train_cfg["project"])

cfg = {k: v for k, v in train_cfg.items() if k != "model"}
n_train = len(list((PROCESSED / "train" / "images").iterdir()))

# get ultralytics MLflow callback from env var
os.environ["MLFLOW_EXPERIMENT_NAME"] = EXPERIMENT
os.environ["MLFLOW_RUN"] = f"{DEVICE}-{n_train}img-{cfg['imgsz']}px-{cfg['epochs']}ep"
os.environ["MLFLOW_KEEP_RUN_ACTIVE"] = "true"

print(f"experiment {EXPERIMENT}")
print(f"run        {os.environ['MLFLOW_RUN']}")

# construct yolo model
model = YOLO(train_cfg["model"])

start = time.time()
# train the model
results = model.train(data=str(data_yaml), **cfg)
elapsed = time.time() - start

print(f"\nelapsed: {elapsed:.0f}s ({elapsed / 60:.1f} min)")

experiment yolo-plate-detection
run        0-445img-640px-10ep


Ultralytics 8.4.119 🚀 Python-3.12.13 torch-2.13.0+cu130 CUDA:0 (Tesla T4, 14913MiB)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/sagemaker-user/sagemaker-yolo/configs/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sagemaker-train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=

Overriding model.yaml nc=80 with nc=1



                   from  n    params  module                                       arguments                     


...

Speed: 0.2ms preprocess, 2.8ms inference, 0.0ms loss, 2.7ms postprocess per image


Results saved to /home/sagemaker-user/sagemaker-yolo/runs/sagemaker-train


MLflow: results logged to arn:aws:sagemaker:ca-central-1:099139718958:mlflow-tracking-server/sagemaker-yolo-dev
MLflow: disable with 'yolo settings mlflow=False'


MLflow: mlflow run still alive, remember to close it using mlflow.end_run()



elapsed: 116s (1.9 min)


### Log dataset context

log hyperparameters and metrics with mlflow.


In [6]:
from notebook.code.tracking import log_dataset_context

# save dir
save_dir = Path(results.save_dir)

# log with mlflow
logged = log_dataset_context(
    PROCESSED,
    RAW,
    **{"data.limit": str(LIMIT), "data.seed": SPLIT_SEED, "run.device": str(DEVICE)},
)
mlflow.log_metric("elapsed_seconds", elapsed)

# get run id
run_id = mlflow.active_run().info.run_id
print(f"run_id {run_id}")
for key, value in logged.items():
    print(f"  {key:26} {value}")

# terminate run
mlflow.end_run()
print("\nrun closed")

run_id d708a0d85ba34b828e58c6cec064bdb4
  data.train_images          445
  data.train_boxes           458
  data.val_images            111
  data.val_boxes             116
  data.total_images          556
  data.available_images      557
  data.fraction_used         0.998
  data.limit                 None
  data.seed                  0
  run.device                 0


🏃 View run 0-445img-640px-10ep at: https://ca-central-1.experiments.sagemaker.aws/#/experiments/1/runs/d708a0d85ba34b828e58c6cec064bdb4
🧪 View experiment at: https://ca-central-1.experiments.sagemaker.aws/#/experiments/1

run closed


Confirm by read run from mlflow.


In [7]:
# get run
fetched = mlflow.get_run(run_id)

print(f"run_id  {run_id}")
print(f"status  {fetched.info.status}")

# print metrics
print("\nfinal metrics")
for key in sorted(fetched.data.metrics):
    if any(m in key for m in ("mAP", "precision", "recall")):
        print(f"  {key:28} {fetched.data.metrics[key]:.4f}")

# print dataset params
print("\ndataset params")
for key in sorted(k for k in fetched.data.params if k.startswith("data.")):
    print(f"  {key:28} {fetched.data.params[key]}")

# print Artifacts
print("\nartifacts")
for artifact in mlflow.artifacts.list_artifacts(run_id=run_id):
    print(f"  {artifact.path}")

run_id  d708a0d85ba34b828e58c6cec064bdb4
status  FINISHED

final metrics
  metrics/mAP50-95B            0.7888
  metrics/mAP50B               0.9854
  metrics/precisionB           0.9734
  metrics/recallB              0.9463

dataset params
  data.available_images        557
  data.fraction_used           0.998
  data.limit                   None
  data.seed                    0
  data.total_images            556
  data.train_boxes             458
  data.train_images            445
  data.val_boxes               116
  data.val_images              111

artifacts


  BoxF1_curve.png
  BoxPR_curve.png
  BoxP_curve.png
  BoxR_curve.png
  args.yaml
  confusion_matrix.png
  confusion_matrix_normalized.png
  labels.jpg
  results.csv
  results.png
  train_batch0.jpg
  train_batch1.jpg
  train_batch2.jpg
  val_batch0_labels.jpg
  val_batch0_pred.jpg
  val_batch1_labels.jpg
  val_batch1_pred.jpg
  val_batch2_labels.jpg
  val_batch2_pred.jpg
  weights


Plot history.


In [8]:
import matplotlib.pyplot as plt

client = mlflow.tracking.MlflowClient()

# mlflow does not guarantee ordering, sort by step before plotting
history = sorted(client.get_metric_history(run_id, "metrics/mAP50-95B"), key=lambda p: p.step)

plt.figure(figsize=(7, 4))
plt.plot([p.step for p in history], [p.value for p in history], marker="o")
plt.xlabel("epoch")
plt.ylabel("mAP50-95")
plt.title("validation mAP50-95")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.show()

print(f"{'epoch':>6} {'mAP50-95':>10}")
for point in history:
    print(f"{point.step:>6} {point.value:>10.4f}")

<Figure size 700x400 with 1 Axes>

 epoch   mAP50-95
     0     0.2881
     1     0.3772
     2     0.6357
     3     0.6293
     4     0.7068
     5     0.7107
     6     0.7545
     7     0.7693
     8     0.7874
     9     0.7885
    10     0.7888


## List runs

Compare every run in the experiment, side by side.


In [9]:
from notebook.code.tracking import compare_runs

compare_runs(EXPERIMENT)

,mlflow.runName,epochs,imgsz,data.train_images,metrics/mAP50B,metrics/mAP50-95B,metrics/precisionB,metrics/recallB
0,0-445img-640px-10ep,10,640,445,0.985369,0.78878,0.973397,0.946302
1,0-445img-640px-10ep,10,640,445,0.985369,0.78878,0.973397,0.946302


Open the MLflow UI from the Studio launcher.

Note the UI lands on the `Default` experiment, which is empty — switch to
`yolo-plate-detection` in the sidebar, or use the direct link printed above.


## Export model

Export `best.pt` to ONNX and upload it to S3.


In [10]:
import shutil

# construct model freom best weights
best = YOLO(str(save_dir / "weights" / "best.pt"))

# export model
exported = Path(best.export(format="onnx", imgsz=train_cfg["imgsz"], opset=12, simplify=True))

# count image
n_images = sum(len(list((PROCESSED / s / "images").iterdir())) for s in ("train", "val"))

# model file name
onnx_path = MODELS / (
    f"{train_cfg['name']}-{DEVICE}-{n_images}img-{train_cfg['imgsz']}px-{train_cfg['epochs']}ep.onnx"
)
shutil.move(str(exported), onnx_path)

print(onnx_path.name)
print(f"{onnx_path.stat().st_size / 1e6:.1f} MB")

Ultralytics 8.4.119 🚀 Python-3.12.13 torch-2.13.0+cu130 CPU (Intel Xeon Platinum 8259CL CPU @ 2.50GHz)


YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs



PyTorch: starting from '/home/sagemaker-user/sagemaker-yolo/runs/sagemaker-train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)



ONNX: starting export with onnx 1.22.0 opset 12...


ONNX: slimming with onnxslim 0.1.95...


2026-08-13 19:46:15.345327560 [W:onnxruntime:Default, device_discovery.cc:285 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:94 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


ONNX: export success ✅ 1.4s, saved as '/home/sagemaker-user/sagemaker-yolo/runs/sagemaker-train/weights/best.onnx' (10.1 MB)



Export complete (1.8s)
Results saved to /home/sagemaker-user/sagemaker-yolo/runs/sagemaker-train/weights/best.onnx
Predict:         yolo predict task=detect model=/home/sagemaker-user/sagemaker-yolo/runs/sagemaker-train/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/home/sagemaker-user/sagemaker-yolo/runs/sagemaker-train/weights/best.onnx imgsz=640 data=/home/sagemaker-user/sagemaker-yolo/configs/data.yaml  
Visualize:       https://netron.app


sagemaker-train-0-556img-640px-10ep.onnx
10.6 MB


Write a metadata file.


In [11]:
import json
from datetime import datetime, timezone

sidecar = onnx_path.with_suffix(".metadata.json")
sidecar.write_text(json.dumps({
    "imgsz": train_cfg["imgsz"],
    "names": [best.names[i] for i in sorted(best.names)],
    "mlflow": {
        "run_id": run_id,
        "experiment": EXPERIMENT,
        "tracking_uri": TRACKING_URI,
    },
    "train": {k: train_cfg[k] for k in ("model", "epochs", "batch", "seed")},
    "images": n_images,
    "exported_at": datetime.now(timezone.utc).isoformat(),
}, indent=2))

print(sidecar.read_text())

{
  "imgsz": 640,
  "names": [
    "car_plate"
  ],
  "mlflow": {
    "run_id": "d708a0d85ba34b828e58c6cec064bdb4",
    "experiment": "yolo-plate-detection",
    "tracking_uri": "arn:aws:sagemaker:ca-central-1:099139718958:mlflow-tracking-server/sagemaker-yolo-dev"
  },
  "train": {
    "model": "yolo11n.pt",
    "epochs": 10,
    "batch": 8,
    "seed": 0
  },
  "images": 556,
  "exported_at": "2026-08-13T19:46:15.920377+00:00"
}


Upload to S3 and attach the same files to the MLflow run.


In [12]:
from notebook.code.s3_sync import list_objects, upload_files

dest = f"{S3_MODELS}/{onnx_path.stem}"
weights = save_dir / "weights" / "best.pt"

# upload model file
for uri in upload_files([onnx_path, sidecar, weights], dest):
    print(uri)

# save export uri to mlflow
with mlflow.start_run(run_id=run_id):
    mlflow.log_artifact(str(onnx_path), artifact_path="export")
    mlflow.log_artifact(str(sidecar), artifact_path="export")
    mlflow.set_tag("export.s3_uri", dest)

print()
for key, meta in sorted(list_objects(dest).items()):
    print(f"  {meta['size'] / 1e6:>6.1f} MB  {key.rsplit('/', 1)[-1]}")

print(f"\nrun: {UI_URL}/#/experiments/{experiment.experiment_id}/runs/{run_id}")

s3://sagemaker-yolo-dev-up68ac/notebook/models/sagemaker-train-0-556img-640px-10ep/sagemaker-train-0-556img-640px-10ep.onnx
s3://sagemaker-yolo-dev-up68ac/notebook/models/sagemaker-train-0-556img-640px-10ep/sagemaker-train-0-556img-640px-10ep.metadata.json
s3://sagemaker-yolo-dev-up68ac/notebook/models/sagemaker-train-0-556img-640px-10ep/best.pt


🏃 View run 0-445img-640px-10ep at: https://ca-central-1.experiments.sagemaker.aws/#/experiments/1/runs/d708a0d85ba34b828e58c6cec064bdb4
🧪 View experiment at: https://ca-central-1.experiments.sagemaker.aws/#/experiments/1

     5.5 MB  best.pt
     0.0 MB  sagemaker-train-0-556img-640px-10ep.metadata.json
    10.6 MB  sagemaker-train-0-556img-640px-10ep.onnx

run: https://t-dwc5gdj0ow9o.ca-central-1.experiments.sagemaker.aws/#/experiments/1/runs/d708a0d85ba34b828e58c6cec064bdb4


## Gate and register

Three checks before anything is registered or promoted:

1. **floor** - `mAP50-95 >= MIN_MAP`, catches a collapsed or truncated run
2. **parity** - the exported ONNX scores within tolerance of the `.pt` weights
3. **promotion** - only a clear win over the current champion moves the alias

A run that fails a gate is tagged `gate.passed=false` with the reason and left
unregistered. Note the floor rejects small `LIMIT` smoke runs by design.


In [13]:
from notebook.code.tracking import MIN_MAP, PROMOTION_MARGIN, check_export_parity, register_model

REGISTERED_MODEL = "yolo-plate-detection"

# score the exported onnx against the .pt metric it was gated on
parity = check_export_parity(
    onnx_path,
    data_yaml,
    reference_map=results.results_dict["metrics/mAP50-95(B)"],
    imgsz=train_cfg["imgsz"],
)
print(f"parity  onnx {parity['onnx_map']:.4f}  pt {parity['reference_map']:.4f}  "
      f"delta {parity['delta']:.4f}  {'ok' if parity['passed'] else 'FAILED'}")

outcome = register_model(
    run_id,
    onnx_path,
    REGISTERED_MODEL,
    min_map=MIN_MAP,
    margin=PROMOTION_MARGIN,
    parity=parity,
)

# print result
print(f"\nscore      {outcome['score']:.4f}  (floor {MIN_MAP})")
if not outcome["registered"]:
    print(f"REJECTED   {outcome['reason']}")
else:
    print(f"registered version {outcome['version']}")
    if outcome["promoted"]:
        print(f"champion   -> version {outcome['version']}")
    else:
        print(f"champion   stays at version {outcome['champion_version']} "
              f"({outcome['champion_score']:.4f})")
    print(f"\nmodel: {UI_URL}/#/models/{REGISTERED_MODEL}/versions/{outcome['version']}")


WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.


Ultralytics 8.4.119 🚀 Python-3.12.13 torch-2.13.0+cu130 CUDA:0 (Tesla T4, 14913MiB)


Loading /home/sagemaker-user/sagemaker-yolo/models/sagemaker-train-0-556img-640px-10ep.onnx for ONNX Runtime inference...


WARNING ⚠️ CUDA requested but CUDAExecutionProvider not available. Using CPU...


Using ONNX Runtime 1.28.0 with CPUExecutionProvider


...

Registered model 'yolo-plate-detection' already exists. Creating a new version of this model...


2026/08/13 19:46:38 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: yolo-plate-detection, version 2


Created version '2' of model 'yolo-plate-detection'.


🏃 View run 0-445img-640px-10ep at: https://ca-central-1.experiments.sagemaker.aws/#/experiments/1/runs/d708a0d85ba34b828e58c6cec064bdb4
🧪 View experiment at: https://ca-central-1.experiments.sagemaker.aws/#/experiments/1



score      0.7888  (floor 0.7)
registered version 2
champion   stays at version 1 (0.7888)

model: https://t-dwc5gdj0ow9o.ca-central-1.experiments.sagemaker.aws/#/models/yolo-plate-detection/versions/2
